In [39]:
import operator
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from IPython.display import Image
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

In [40]:
load_dotenv()  # Load environment variables from .env file

True

In [41]:
class BatsManState(TypedDict):
    
    runs: int
    balls: int
    fours: int
    sixes: int
    
    strike_rate: float
    balls_per_boundary: float   
    boundary_percentage: float
    summary: str

In [42]:
def calculate_strike_rate(state: BatsManState) -> BatsManState:
    state['strike_rate'] = (state['runs'] / state['balls']) * 100
    return {'strike_rate': state['strike_rate']}

In [43]:
def calculate_balls_per_boundary(state: BatsManState) -> BatsManState:
    total_boundaries = state['fours'] + state['sixes']
    if total_boundaries > 0:
        state['balls_per_boundary'] = state['balls'] / total_boundaries
    return {'balls_per_boundary': state.get('balls_per_boundary', 0)}

In [44]:
def calculate_boundary_percentage(state: BatsManState) -> BatsManState:
    if state['runs'] > 0:
        state['boundary_percentage'] = (((state['fours'] * 4) + (state['sixes'] * 6)) / state['runs'])* 100
    return {'boundary_percentage': state.get('boundary_percentage', 0)}

In [45]:
def summary(state: BatsManState) -> BatsManState:
    summary = f"""
    Batsman scored {state['runs']} runs in {state['balls']} balls, hitting {state['fours']} fours and {state['sixes']} sixes.
    The strike rate is {state['strike_rate']:.2f}, balls per boundary is {state['balls_per_boundary']:.2f}, and boundary percentage is {state['boundary_percentage']:.2f}%.
    """
    state['summary'] = summary
    return {'summary': state['summary']}

In [46]:
graph = StateGraph(BatsManState)
graph.add_node('calculate_strike_rate',calculate_strike_rate)
graph.add_node('calculate_balls_per_boundary',calculate_balls_per_boundary)
graph.add_node('calculate_boundary_percentage',calculate_boundary_percentage)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_balls_per_boundary')
graph.add_edge(START, 'calculate_boundary_percentage')

graph.add_edge('calculate_strike_rate', 'summary')
graph.add_edge('calculate_balls_per_boundary', 'summary')
graph.add_edge('calculate_boundary_percentage', 'summary')
graph.add_edge('summary', END)

In [47]:
workflow = graph.compile()

In [48]:
initial_state = {
    'runs': 100,
    'balls': 60,
    'fours': 10,
    'sixes': 5
}

final_state = workflow.invoke(initial_state)
print(final_state['summary'])


    Batsman scored 100 runs in 60 balls, hitting 10 fours and 5 sixes.
    The strike rate is 166.67, balls per boundary is 4.00, and boundary percentage is 70.00%.
    


## UPSC Essay Evaluation workflow

In [49]:
model = ChatOpenAI(model='gpt-4o-mini')

In [50]:
class EvaluationSchema(BaseModel):
    
    feedback: str = Field(description="Detailed feedback on the essay")
    score: int = Field(description="Score out of 10 for the essay", ge=0, le=10)

In [51]:
structured_model = model.with_structured_output(EvaluationSchema)

In [52]:
essay = f"""India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

In [53]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).feedback

"The essay presents a coherent and insightful analysis of India's potential and challenges in the age of AI. The structure is well-organized, transitioning smoothly from India's strengths to its challenges and concluding with a forward-looking perspective. The language is formal and academic, suitable for the topic, while also being engaging. There are some areas for improvement, such as providing specific examples or data to support claims and enhancing the depth of analysis in certain sections, particularly regarding the implications of the digital divide and job displacement. A more critical examination of the risks connected with AI could further enhance the discussion. Overall, the essay is well-articulated but would benefit from more empirical support and nuanced exploration of the complexities around AI in India."

In [54]:
class UPSCState(TypedDict):
    
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [55]:
def evaluate_language(state: UPSCState) -> UPSCState:
    
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    evaluation = structured_model.invoke(prompt)
    state['language_feedback'] = evaluation.feedback
    state['individual_score'] = [evaluation.score]
    return {'language_feedback': state['language_feedback'], 'individual_score': state['individual_score']}

In [56]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    
    prompt = f'Evaluate the analysis quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    evaluation = structured_model.invoke(prompt)
    state['analysis_feedback'] = evaluation.feedback
    state['individual_score'] = [evaluation.score]
    return {'analysis_feedback': state['analysis_feedback'], 'individual_score': state['individual_score']}

In [57]:
def evaluate_clarity(state: UPSCState) -> UPSCState:
    
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    evaluation = structured_model.invoke(prompt)
    state['clarity_feedback'] = evaluation.feedback
    state['individual_score'] = [evaluation.score]
    return {'clarity_feedback': state['clarity_feedback'], 'individual_score': state['individual_score']}

In [58]:
def final_evaluation(state: UPSCState) -> UPSCState:
    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    evaluation = model.invoke(prompt).content
    
    # avg score
    avg_score = sum(state['individual_score'])/len(state['individual_score'])
    return {'overall_feedback': evaluation, 'avg_score': avg_score}


In [59]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_clarity',evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

In [60]:
workflow = graph.compile()

In [63]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [67]:
intial_state = {'essay': essay2}
workflow.invoke(intial_state)

{'essay': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.\n\n